# Task 2: From Clean EDA to a Model-Ready Dataset

Task 1 explored the NYC Uber dataset. This turns it into a dataset ready for a **regression model that predicts fare_amount**, following the 10 required steps in order: **missing values → duplicates → encoding → scaling → feature extraction → feature selection → outliers → split → cross-validation → tuning.**




In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
DATA_PATH = '/content/final_internship_data.csv'
raw = pd.read_csv(DATA_PATH)
df = raw.copy()
print(df.shape)

(500000, 26)


## Before Step 1: carrying over two fixes from Task 1


- **Column names and three ID columns.** Task 1 states User ID, User Name and Driver Name identify who, not anything about the fare, so they are dropped immediately.
- **Coordinates were stored in radians, not degrees.**

In [ ]:
df.columns = (df.columns.str.strip().str.lower().str.replace(' ', '_'))
df = df.rename(columns={'day': 'day_of_month', 'weekday': 'day_of_week'})
df = df.drop(columns=['user_id', 'user_name', 'driver_name'])

coord_cols = ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude']  # all four, stored in radians
df[coord_cols] = np.degrees(df[coord_cols])        # convert radians -> degrees, as
df.shape

(500000, 23)

## 1. Handling Missing Values



In [ ]:
missing_pct = df.isnull().mean() * 100
print(missing_pct[missing_pct > 0].round(4))        # only show columns that actually have missing values

dropoff_longitude    0.001
dropoff_latitude     0.001
jfk_dist             0.001
ewr_dist             0.001
lga_dist             0.001
sol_dist             0.001
nyc_dist             0.001
distance             0.001
bearing              0.001
dtype: float64


In [ ]:
df = df.dropna()

in_nyc = (df['pickup_longitude'].between(-74.3, -73.7) &      # pickup inside a box around NYC
          df['pickup_latitude'].between(40.5, 41.0) &
          df['dropoff_longitude'].between(-74.3, -73.7) &     # dropoff inside the same box
          df['dropoff_latitude'].between(40.5, 41.0))
valid_fare = df['fare_amount'].between(2.5, 100)
valid_passengers = df['passenger_count'].between(1, 6) # 0 passengers isn't a ride; a cab holds at most 6
valid_trip = df['distance'] >= 0.05                    # pickup ~= dropoff isn't a real trip

df = df[in_nyc & valid_fare & valid_passengers & valid_trip]   # treat all four as 'effectively missing' (collection errors) and drop them
print(f'Rows remaining: {len(df):,} ({len(df) / len(raw):.1%} of the original {len(raw):,})')

Rows remaining: 479,944 (96.0% of the original 500,000)


## 2. Detecting and Removing Duplicates


In [ ]:
exact_dupes = df.duplicated().sum()                 # rows identical across every single column
key_dupes = df['key'].duplicated().sum()             # the same trip ID appearing more than once

print('Exact duplicate rows:', exact_dupes)
print('Duplicate trip keys :', key_dupes)

df = df.drop_duplicates()
df = df[~df['key'].duplicated()]
df = df.drop(columns=['key'])                         # it says it's for deduplication only so it's done, so drop it
print(f'Rows remaining: {len(df):,}')

Exact duplicate rows: 0
Duplicate trip keys : 0
Rows remaining: 479,944


Both checks came back at **0** — this dataset has no exact or key-based duplicates.

## 3. Encoding Categorical Variables

Three categorical columns, two different encodings, because they're not the same *kind* of category:

- **car_condition** and **traffic_condition**  both have a **real order** so use **ordinal encoding**.
- **weather**  has **no natural order** and only 5 values, so use **one-hot encoding**.

Both are built as ColumnTransformer steps later in Section 4, not applied to df, so that they can later be **fit on the training data only**.

In [ ]:
car_order = {'Bad': 0, 'Good': 1, 'Very Good': 2, 'Excellent': 3}       # ordinal map: worst to best
traffic_order = {'Flow Traffic': 0, 'Dense Traffic': 1, 'Congested Traffic': 2}   # ordinal map: lightest to heaviest

df['car_condition_ord'] = df['car_condition'].map(car_order)
df['traffic_condition_ord'] = df['traffic_condition'].map(traffic_order)

df[['car_condition', 'car_condition_ord', 'traffic_condition', 'traffic_condition_ord']].head()

,car_condition,car_condition_ord,traffic_condition,traffic_condition_ord
0,Very Good,2,Congested Traffic,2
1,Excellent,3,Flow Traffic,0
2,Bad,0,Congested Traffic,2
3,Very Good,2,Flow Traffic,0
4,Bad,0,Congested Traffic,2


## 4. Feature Scaling



In [ ]:
numeric_cols = ['distance', 'jfk_km', 'passenger_count', 'year', 'hour_sin', 'hour_cos',   # ordinal numeric features
                'car_condition_ord', 'traffic_condition_ord']
cat_cols = ['weather']                               # one-hot encoded
passthrough_cols = ['is_weekend', 'is_near_airport']

# It is not fit yet, it only gets fit once, inside the Pipeline, on the training data.

## 5. Creating New Features

Three new features :

- **hour_sin/ hour_cos** In Task 1 (Q3) found fares peak sharply at 5 AM — hour 23 and hour 0 are only 1 hour apart in reality, but a plain integer 0–23 makes them look 23 apart. Encoding hour as sin/cos of its position on a 24-hour circle fixes that.
- **is_weekend**
- **is_near_airport**  Task 1 (Q8) the strongest effect in the whole dataset, pickups within 2 km of JFK or LGA average far more than the rest of NYC. The raw file's jfk_dist/lga_dist columns were found in Task 1 to be **2x the true distance**, so they are not reused so jfk_km is recomputed here with the same haversine formula from Task 1, and lga_km is added too.


In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between two points (same formula used in Task 1)."""
    R = 6371.0                                                     # Earth's radius in km
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = phi2 - phi1
    dlmb = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlmb / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

df['jfk_km'] = haversine_km(df['pickup_latitude'], df['pickup_longitude'], 40.6413, -73.7781)   # corrected distance to JFK
df['lga_km'] = haversine_km(df['pickup_latitude'], df['pickup_longitude'], 40.7769, -73.8740)   # corrected distance to LGA
df['is_near_airport'] = ((df['jfk_km'] < 2) | (df['lga_km'] < 2)).astype(int)                   # pickup within 2km of either airport

df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df[['jfk_km', 'lga_km', 'is_near_airport', 'is_weekend', 'hour_sin', 'hour_cos']].head()

,jfk_km,lga_km,is_near_airport,is_weekend,hour_sin,hour_cos
0,10.504346,6.667174,0,0,-0.965926,-0.258819
1,21.523172,14.014555,0,0,-0.866025,-0.500000
2,21.807027,9.320644,0,0,0.000000,1.000000
3,20.369738,10.699370,0,1,0.866025,0.500000
4,21.330886,7.985091,0,0,0.965926,-0.258819


## 6. Feature Selection

**Two layers, one manual and one automatic, matching the material's own distinction between the two:**

**Manual:** Task 1 confirmed the file's own jfk_dist/lga_dist are double the distance and checking correlations among them shows ewr_dist, sol_dist and nyc_dist correlate with each other Rather than pick one of five columns, all five are dropped in favor of the two (jfk_km, lga_km).

**Automatic:** SelectFromModel(Lasso(...)) is added as a Pipeline step, after the encoder/ and before the final regression model. Because it lives inside the Pipeline, it gets refit on each individual training fold during cross-validation.

In [ ]:
df = df.drop(columns=['jfk_dist', 'lga_dist', 'ewr_dist', 'sol_dist', 'nyc_dist', 'bearing'])

feature_cols = ['distance', 'jfk_km', 'passenger_count', 'year', 'hour_sin', 'hour_cos',
                'is_weekend', 'is_near_airport', 'car_condition_ord', 'traffic_condition_ord', 'weather']
X = df[feature_cols].copy()
y = df['fare_amount'].copy()
print(X.shape, y.shape)

(479944, 11) (479944,)


## 7. Outlier Detection & Treatment

Task 1 already found two *different kinds* of extreme value in this dataset, and they're treated differently on purpose:

- **Collection errors** (negative fares, 0 passengers, GPS outside NYC) were already removed in Section 1, because they are mistakes, not real data.
- **Real, rare values** trips up to 40 km, and airport fares around $45–57 are valid rides, not errors (Task 1 confirmed the airport fares in Q8). Deleting them would teach the model that long trips don't exist. Instead:
  - A small Winsorizer step is added to the Pipeline, fit on the training data.


In [ ]:
class Winsorizer(BaseEstimator, TransformerMixin):
    """Caps the 'distance' column (always column 0 in numeric_cols) at a quantile LEARNED FROM TRAINING DATA ONLY."""
    def __init__(self, upper_q=0.99):
        self.upper_q = upper_q

    def fit(self, X, y=None):
        X = np.asarray(X)
        self.cap_ = np.quantile(X[:, 0], self.upper_q)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float).copy()
        X[:, 0] = np.minimum(X[:, 0], self.cap_)
        return X



## 8. Train/Test Split and Avoiding Data Leakage



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)   # 80/20 split
print(f'Train: {X_train.shape[0]:,} rows   Test: {X_test.shape[0]:,} rows')

numeric_pipeline = Pipeline([
    ('cap_outliers', Winsorizer(upper_q=0.99)),
    ('scale', StandardScaler()),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols),         # S one-hot for weather
    ('pass', 'passthrough', passthrough_cols),
])

baseline_pipeline = Pipeline([
    ('preprocess', preprocessor),                                            # encoding + scaling + outlier capping
    ('select', SelectFromModel(Lasso(alpha=0.01, max_iter=5000))),
    ('model', LinearRegression()),
])

baseline_model = TransformedTargetRegressor(
    regressor=baseline_pipeline, func=np.log1p, inverse_func=np.expm1,
)

Train: 383,955 rows   Test: 95,989 rows


## 9. Cross-Validation

KFold with k=5, 5-fold (rather than 10) keeps the fit time reasonable on 384,000 training rows while still giving a stable estimate.

Cross-validation is run only on the training set, untouched by training

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_mae = -cross_val_score(
    baseline_model, X_train, y_train, cv=cv,
    scoring='neg_mean_absolute_error', n_jobs=-1,
)
print(f'Baseline 5-fold CV MAE: {cv_mae.mean():.3f}  (+/- {cv_mae.std():.3f})')

Baseline 5-fold CV MAE: 2.951  (+/- 0.019)


## 10. Hyperparameter Tuning

**Model:** Ridge regression so the comparison in the results table isolates the effect of tuning.

**Technique:** RandomizedSearchCV over Ridge's strength (alpha)

 test set still hasn't been touched.

In [ ]:
tuned_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('select', SelectFromModel(Lasso(alpha=0.01, max_iter=5000))),
    ('model', Ridge()),                                                  # Ridge
])
tuned_model = TransformedTargetRegressor(regressor=tuned_pipeline, func=np.log1p, inverse_func=np.expm1)

param_distributions = {
    'regressor__model__alpha': np.logspace(-3, 3, 50),    # 50 candidate values from 0.001 to 1000, log-spaced
}

search = RandomizedSearchCV(
    tuned_model, param_distributions=param_distributions,
    n_iter=15, cv=cv, scoring='neg_mean_absolute_error',
    random_state=42, n_jobs=-1,
)
search.fit(X_train, y_train)

print('Best alpha:', search.best_params_['regressor__model__alpha'])
print(f'Best CV MAE: {-search.best_score_:.3f}')

Best alpha: 754.3120063354608
Best CV MAE: 2.945


## Results: Baseline vs. Tuned

Both models are fit once on the full training set, then scored on X_test/y_test.

In [ ]:
baseline_model.fit(X_train, y_train)
baseline_preds = baseline_model.predict(X_test)

best_model = search.best_estimator_
tuned_preds = best_model.predict(X_test)

results = pd.DataFrame({
    'Model': ['Baseline (Linear Regression)', 'Tuned (Ridge, alpha=%.1f)' % search.best_params_['regressor__model__alpha']],
    'MAE':  [mean_absolute_error(y_test, baseline_preds), mean_absolute_error(y_test, tuned_preds)],
    'RMSE': [mean_squared_error(y_test, baseline_preds) ** 0.5, mean_squared_error(y_test, tuned_preds) ** 0.5],
    'R2':   [r2_score(y_test, baseline_preds), r2_score(y_test, tuned_preds)],
})
results.round(3)

,Model,MAE,RMSE,R2
0,Baseline (Linear Regression),2.945,6.615,0.483
1,"Tuned (Ridge, alpha=754.3)",2.940,6.572,0.490


**Reading the table:** MAE is the average fare prediction error in real dollars. RMSE is also in dollars but penalizes big misses harder. R2 is the share of fare variation the model explains (1.0 = perfect, 0.0 = no better than guessing the average). The Lasso-based feature selection already removed the main source of noise (weather, traffic, car condition, passenger count) so there isn't overfitting left for Ridge's regularization to fix.

In [ ]:
joblib.dump(best_model, 'fare_prediction_pipeline.joblib')     # saves preprocessing + feature selection + tuned model as one file
print('Saved fare_prediction_pipeline.joblib')

Saved fare_prediction_pipeline.joblib


### What I would try next

- **Try target/frequency encoding** instead of one-hot for weather, and check whether the Lasso-based selector keeps it this time.
- **Try Random Forest regressor** in place of Ridge. Tree-based models are more forgiving between distance and jfk_km, and could capture the non-linear in airport fares (Q8).
- **Add trip duration as a feature**, if it can be approximated from distance and known average NYC traffic speed by hour thefare in real taxi pricing is a function of time as well as distance.